### SVG generation & evaluation workflow

In [1]:
### Init...
### df load
### model call vllm
### get df response
### close vllm server
### clean df response
### call siglip
### get score
### close siglip server 
### write output
### new_metric_score (manual)

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
import torch
import gc
import time
sys.path.append('./utils')
sys.path.append('/home/vino/.cache/huggingface/hub')

### Define

In [3]:
model_path="models--unsloth--Llama-3.2-1B-Instruct"
global model_path

In [4]:
from vllm import LLM, SamplingParams
from vllm_model_server_start import vllm_model_server_start
from start_siglip_server import start_siglip_server
from siglip_class import SVGMetricEvaluator
from terminate_server import terminate_server

INFO 04-20 23:24:47 [__init__.py:239] Automatically detected platform cuda.


### Start vLLM model server

In [ ]:
vllm_process= vllm_model_server_start(model_path)

### OpenAI style client

In [ ]:
from openai import OpenAI
# Set OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "my-api-key"
openai_api_base = "http://localhost:8000/v1"

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

def get_reponse(description):
    alpaca_prompt = f"""Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

            ### Instruction:
            Generate a SVG code for the given input:
            
            ### Input:
            {description}
                            
            ### Response:
            """
    
    formatted_input = alpaca_prompt.format(description)
    chat_response = client.chat.completions.create(
        model=model_path,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": f"{formatted_input}"},
            ],
        temperature=0.6,
        top_p=0.95,
        seed=123
    )
    return chat_response.choices[0].message.content


### Load df

In [ ]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test_vqa.csv',header=[0])
print(df.shape)
df.head(2)


### Concurrent calls to vLLM server 

In [ ]:
import time
from tqdm import tqdm
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# Wrap tqdm over futures
def parallel_apply_with_tqdm(func, data, max_workers=8):
    results = [None] * len(data)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(func, data[i]): i for i in range(len(data))}
        for future in tqdm(as_completed(futures), total=len(data)):
            idx = futures[future]
            try:
                results[idx] = future.result()
            except Exception as e:
                results[idx] = None
                print(f"Error at index {idx}: {e}")
    return results

# Example usage
start_time = time.time()
df['response_3'] = parallel_apply_with_tqdm(get_reponse, df['description'].tolist(), max_workers=8)

end_time = time.time()
print(f"Total time taken: {end_time - start_time:.2f} seconds")


### Terminate vLLM Server

In [ ]:
terminate_server(vllm_process)

### Extract csv from response & clean csv

In [ ]:
from clean_svg_response import Model
from tqdm import tqdm
tqdm.pandas()
model=Model()
df['svg_3'] = df.progress_apply(lambda row: model.clean_svg(row['response_3']), axis=1)

### Start sl server

In [ ]:
server_process=start_siglip_server()

### Get sl score from sl server (sometime getting slow, need to check why...)

In [ ]:
import pandas as pd
import httpx
from tqdm import tqdm

API_URL = "http://127.0.0.1:8000/evaluate_svg"
API_KEY = "my-api-key"

def send_request(client, prompt, svg):
    try:
        response = client.post(
            API_URL,
            headers={"x-api-key": API_KEY},
            json={"prompt": prompt, "svg": svg},
            timeout=10.0
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e)}

def evaluate_all(df):
    with httpx.Client() as client:
        results = []
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating SVGs"):
            result = send_request(client, row["description"], row["svg_3"])
            results.append(result)
        return results

# Run evaluation
results = evaluate_all(df)

# Append score or error
df["score_3"] = [r.get("score") if "score" in r else r.get("error") for r in results]


### Get sl score from sl fn

In [ ]:
# from tqdm import tqdm
# tqdm.pandas()
# evaluator=SVGMetricEvaluator()
# df['score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['csv_3']), axis=1)

### Terminate sl server

In [ ]:
terminate_server(server_process)

In [ ]:
#df = df[~df['score_3'].apply(lambda x: isinstance(x, str))]
print(df.shape)
print(df['score_3'].mean())

In [ ]:
df.to_csv('Llama-3.2-3B-Instruct_r256_s2000_i1000_v1.csv',index=False)

In [ ]:
df